In [1]:
import pandas as pd

def fix_old_data():

    df = pd.read_csv('../data_logs.csv')

    df['timestamp'] = pd.to_datetime(df['timestamp'])

    df = df.sort_values(by='timestamp').reset_index(drop=True)
    
    time_diff = df['timestamp'].diff().dt.total_seconds()

    new_session_mask = (time_diff > 120) 
    
    
    df['session_id'] = new_session_mask.cumsum() + 1
    df['session_id'] = "Session_" + df['session_id'].astype(str)

    
    session_counts = df['session_id'].value_counts()
    short_sessions = session_counts[session_counts < 10].index
    df = df[~df['session_id'].isin(short_sessions)]

  
    df['sec_since_start'] = df.groupby('session_id')['timestamp'].transform(
        lambda x: (x - x.iloc[0]).dt.total_seconds()
    )

    
    df['is_clogged'] = 1

  
    df['cpu_temp_slope'] = df.groupby('session_id')['cpu_temp_C'].diff().fillna(0)
    df['gpu_temp_slope'] = df.groupby('session_id')['gpu_temp_C'].diff().fillna(0)

    
    cols = [
        "timestamp", "session_id", "sec_since_start", "is_clogged",
        "activity", "surface_type", "cpu_boost_mode", 
        "cpu_temp_C", "cpu_temp_slope", "cpu_power_W", "cpu_util_pct", "cpu_freq_MHz", 
        "ram_used_GB", "gpu_temp_C", "gpu_temp_slope", "gpu_util_pct", "gpu_clock_MHz", "gpu_power_W"
    ]
    
   
    final_cols = [c for c in cols if c in df.columns]
    df = df[final_cols]

    
    df.to_csv("../data_logs.csv", index=False)
   
    print(f"Detected {df['session_id'].nunique()} distinct sessions in your old data.")


fix_old_data()

Detected 11 distinct sessions in your old data.
